# Neural Network — MNIST Digit Classifier

We'll build a simple feedforward neural network from scratch using PyTorch to classify handwritten digits (0–9).

**What you'll learn:**
- How to load and normalize data
- How to define a neural network
- How to train with a loss function and optimizer
- How to evaluate accuracy on test data

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

print('PyTorch version:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 1. Load the Data

MNIST has 60,000 training images and 10,000 test images of handwritten digits.
Each image is 28×28 pixels in grayscale.

In [ ]:
# Normalize pixel values from [0, 255] to [-1, 1]
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.MNIST(root='data', train=True,  download=True, transform=transform)
test_data  = datasets.MNIST(root='data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False)

print(f'Training samples: {len(train_data)}')
print(f'Test samples:     {len(test_data)}')

## 2. Visualize Some Samples

In [ ]:
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap='gray')
    ax.set_title(labels[i].item())
    ax.axis('off')
plt.suptitle('Sample MNIST Images', y=1.02)
plt.tight_layout()
plt.show()

## 3. Define the Neural Network

Architecture:
```
Input (784)  →  Hidden Layer 1 (256)  →  ReLU
             →  Hidden Layer 2 (128)  →  ReLU
             →  Output (10 classes)
```
- **784** = 28×28 pixels flattened
- **ReLU** activation adds non-linearity so the network can learn complex patterns
- **10 outputs** = one score per digit (0–9)

In [ ]:
class NeuralNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),           # 28x28 → 784
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)      # 10 digit classes
        )

    def forward(self, x):
        return self.network(x)

model = NeuralNet().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {total_params:,}')

## 4. Set Up Loss Function & Optimizer

- **CrossEntropyLoss** — standard loss for classification; measures how wrong our predictions are
- **Adam** — adaptive optimizer, generally works well out of the box

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## 5. Train the Model

In [ ]:
EPOCHS = 5
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()        # clear old gradients
        outputs = model(images)      # forward pass
        loss = criterion(outputs, labels)  # compute loss
        loss.backward()              # backpropagation
        optimizer.step()             # update weights

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f'Epoch {epoch+1}/{EPOCHS}  |  Loss: {avg_loss:.4f}')

print('\nTraining complete!')

## 6. Plot Training Loss

In [ ]:
plt.plot(range(1, EPOCHS+1), train_losses, marker='o')
plt.title('Training Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

## 7. Evaluate on Test Set

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():  # no gradient needed for evaluation
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Test Accuracy: {accuracy:.2f}%')

## 8. See What the Model Predicts

In [ ]:
model.eval()
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].cpu().squeeze(), cmap='gray')
    color = 'green' if predicted[i] == labels[i] else 'red'
    ax.set_title(f'P:{predicted[i].item()} T:{labels[i].item()}', color=color)
    ax.axis('off')
plt.suptitle('Predictions (green=correct, red=wrong)', y=1.02)
plt.tight_layout()
plt.show()

## Next Steps

- Add **Dropout** layers to reduce overfitting
- Try a **Convolutional Neural Network (CNN)** — much better for images
- Experiment with different learning rates, batch sizes, or architectures
- Try a different dataset (CIFAR-10, FashionMNIST)